In [ ]:
# 从纯文本导入 英文 分节经文 到数据库

import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

def parse_filename(filename):
    name = os.path.splitext(filename)[0].replace("_en", "")
    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)
    chapter = int(match.group(2))
    book_id = get_book_id(book_abbr)
    return book_abbr, chapter, book_id


def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_en FROM verse WHERE id = ?", (verse_id,)
        )
        row = cur.fetchone()

        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            print(f"➕ 新增 {verse_id}")

        elif row[0] == text_en:
            pass

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_en = ? WHERE id = ?",
                    (text_en, verse_id)
                )
                print(f"♻️  已覆盖 {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 导入完成")


# 只问文件名
if __name__ == "__main__":
    filename = input("请输入文件名（如 Mt.1.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

In [11]:
# import_cn.py
# 从纯文本导入中文分节经文（text_cn）
# 文件名规范：2K_4_cn.txt

import sqlite3
import os
import re

# ==========================
# 数据库连接
# ==========================
conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ==========================
# 从 book 表获取 book_id
# ==========================
def get_book_id(abbr_en):
    cur = cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cur.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]

# ==========================
# 解析文件名（2K_4_cn.txt）
# ==========================
def parse_filename(filename):
    name = os.path.splitext(filename)[0]          # 去掉 .txt
    name = name.replace("_cn", "")                 # 去掉 _cn

    match = re.match(r"([A-Za-z0-9]+)\_(\d+)", name)
    if not match:
        raise ValueError(f"文件名格式错误：{filename}")

    book_abbr = match.group(1)   # 2K
    chapter = int(match.group(2))  # 4
    book_id = get_book_id(book_abbr)

    return book_abbr, chapter, book_id

# ==========================
# 导入中文经文
# ==========================
def import_verses_from_file(filepath):
    book_abbr, chapter, book_id = parse_filename(os.path.basename(filepath))

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    for i, text_cn in enumerate(lines, start=1):
        verse_id = f"{book_abbr}.{chapter}.{i}"

        cur = cursor.execute(
            "SELECT text_cn FROM verse WHERE id = ?",
            (verse_id,)
        )
        row = cur.fetchone()

        # verse 不存在（极少见）
        if row is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, "", text_cn))
            print(f"➕ 新增 {verse_id}")

        elif row[0] is None or row[0] == "":
            cursor.execute(
                "UPDATE verse SET text_cn = ? WHERE id = ?",
                (text_cn, verse_id)
            )
            print(f"➕ 写入 text_cn: {verse_id}")

        else:
            ans = input(f"\n⚠️  已存在 {verse_id}，是否覆盖中文译文？(Y/N): ").strip().upper()
            if ans == "Y":
                cursor.execute(
                    "UPDATE verse SET text_cn = ? WHERE id = ?",
                    (text_cn, verse_id)
                )
                print(f"♻️  已覆盖 text_cn: {verse_id}")
            else:
                print(f"⏭️  跳过 {verse_id}")

    conn.commit()
    print(f"\n✅ {book_abbr}.{chapter} 中文经文导入完成")

# ==========================
# 主程序
# ==========================
if __name__ == "__main__":
    filename = input("请输入中文经文文件名（如 2K_4_cn.txt）：").strip()
    filepath = os.path.join("outputs", "plaintext", filename)

    if not os.path.exists(filepath):
        print(f"❌ 文件不存在：{filepath}")
    else:
        import_verses_from_file(filepath)

请输入中文经文文件名（如 2K_4_cn.txt）：Mt_10_cn.txt
➕ 写入 text_cn: Mt.10.1
➕ 写入 text_cn: Mt.10.2
➕ 写入 text_cn: Mt.10.3
➕ 写入 text_cn: Mt.10.4
➕ 写入 text_cn: Mt.10.5
➕ 写入 text_cn: Mt.10.6
➕ 写入 text_cn: Mt.10.7
➕ 写入 text_cn: Mt.10.8
➕ 写入 text_cn: Mt.10.9
➕ 写入 text_cn: Mt.10.10
➕ 写入 text_cn: Mt.10.11
➕ 写入 text_cn: Mt.10.12
➕ 写入 text_cn: Mt.10.13
➕ 写入 text_cn: Mt.10.14
➕ 写入 text_cn: Mt.10.15
➕ 写入 text_cn: Mt.10.16
➕ 写入 text_cn: Mt.10.17
➕ 写入 text_cn: Mt.10.18
➕ 写入 text_cn: Mt.10.19
➕ 写入 text_cn: Mt.10.20
➕ 写入 text_cn: Mt.10.21
➕ 写入 text_cn: Mt.10.22
➕ 写入 text_cn: Mt.10.23
➕ 写入 text_cn: Mt.10.24
➕ 写入 text_cn: Mt.10.25
➕ 写入 text_cn: Mt.10.26
➕ 写入 text_cn: Mt.10.27
➕ 写入 text_cn: Mt.10.28
➕ 写入 text_cn: Mt.10.29
➕ 写入 text_cn: Mt.10.30
➕ 写入 text_cn: Mt.10.31
➕ 写入 text_cn: Mt.10.32
➕ 写入 text_cn: Mt.10.33
➕ 写入 text_cn: Mt.10.34
➕ 写入 text_cn: Mt.10.35
➕ 写入 text_cn: Mt.10.36
➕ 写入 text_cn: Mt.10.37
➕ 写入 text_cn: Mt.10.38
➕ 写入 text_cn: Mt.10.39
➕ 写入 text_cn: Mt.10.40
➕ 写入 text_cn: Mt.10.41
➕ 写入 text_cn: Mt.10.42

✅ M

In [ ]:
# 从纯文本导入英文分节经文到数据库（按书卷缩写）

import sqlite3
import os
import re

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

# ========== 配置 ==========
PLAINTEXT_DIR = "outputs/plaintext"

# ========== 工具函数 ==========
def get_book_info(abbr_en):
    cursor.execute(
        "SELECT id, name_en, max_chapter FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row  # (id, name_en, max_chapter)


def find_chapter_files(abbr_en, max_chapter):
    """
    在 outputs/plaintext 中查找：
    abbr_en_1_en.txt ~ abbr_en_{max_chapter}_en.txt
    """
    files = []
    for ch in range(1, max_chapter + 1):
        filename = f"{abbr_en}_{ch}_en.txt"
        filepath = os.path.join(PLAINTEXT_DIR, filename)
        if os.path.exists(filepath):
            files.append((ch, filepath))
        else:
            print(f"⚠️ 缺少文件：{filename}")
    return files


def import_chapter(book_id, chapter, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]

    inserted = 0

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{book_id}.{chapter}.{i}"

        cursor.execute(
            "SELECT 1 FROM verse WHERE id = ?",
            (verse_id,)
        )

        if cursor.fetchone() is None:
            cursor.execute("""
                INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
                VALUES (?, ?, ?, ?, ?, ?)
            """, (verse_id, book_id, chapter, i, text_en, ""))
            inserted += 1

    conn.commit()
    return inserted


# ========== 主流程 ==========
if __name__ == "__main__":
    abbr_en = input("请输入书卷缩写（如 Gen、Matt）：").strip()

    try:
        book_id, name_en, max_chapter = get_book_info(abbr_en)
        print(f"\n📖 书卷：{name_en}（{abbr_en}），共 {max_chapter} 章")

        chapters = find_chapter_files(abbr_en, max_chapter)

        if not chapters:
            print("❌ 没有找到任何可导入的章节文件")
        else:
            total_inserted = 0
            for chapter, filepath in chapters:
                count = import_chapter(book_id, chapter, filepath)
                print(f"  ✅ {abbr_en} {chapter}：新增 {count} 节")
                total_inserted += count

            print(f"\n🎉 导入完成！共新增 {total_inserted} 节经文")

    except Exception as e:
        print(e)

In [ ]:
import sqlite3
import os

# ========== 配置 ==========
DB_PATH = "db/bible.db"
PLAINTEXT_DIR = "outputs/plaintext"

# ========== 数据库连接 ==========
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# ---------- 确保进度表 ----------
cursor.execute("""
CREATE TABLE IF NOT EXISTS import_progress (
    abbr_en TEXT,
    chapter INTEGER,
    done INTEGER DEFAULT 0,
    PRIMARY KEY (abbr_en, chapter)
)
""")
conn.commit()

# ========== 工具函数 ==========
def get_book_info(abbr_en):
    cursor.execute(
        "SELECT id, name_en, max_chapter FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 未找到书卷：{abbr_en}")
    return row


def is_chapter_imported(abbr_en, chapter):
    cursor.execute(
        "SELECT 1 FROM import_progress WHERE abbr_en = ? AND chapter = ? AND done = 1",
        (abbr_en, chapter)
    )
    return cursor.fetchone() is not None


def mark_chapter_done(abbr_en, chapter):
    cursor.execute("""
        INSERT INTO import_progress (abbr_en, chapter, done)
        VALUES (?, ?, 1)
        ON CONFLICT(abbr_en, chapter) DO UPDATE SET done = 1
    """, (abbr_en, chapter))
    conn.commit()


def find_chapter_files(abbr_en, max_chapter):
    files = []
    for ch in range(1, max_chapter + 1):
        filename = f"{abbr_en}_{ch}_en.txt"
        filepath = os.path.join(PLAINTEXT_DIR, filename)
        if os.path.exists(filepath):
            files.append((ch, filepath))
        else:
            print(f"⚠️ 缺少文件：{filename}")
    return files


def import_chapter(book_id, abbr_en, chapter, filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    inserted = 0
    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{abbr_en}.{chapter}.{i}"
        cursor.execute("SELECT 1 FROM verse WHERE id = ?", (verse_id,))
        if cursor.fetchone():
            continue

        cursor.execute("""
            INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (verse_id, book_id, chapter, i, text_en, ""))
        inserted += 1

    conn.commit()
    mark_chapter_done(abbr_en, chapter)
    return inserted

# ========== 主流程 ==========
if __name__ == "__main__":
    abbr_en = input("请输入书卷缩写（如 Gen、Matt）：").strip()

    try:
        book_id, name_en, max_chapter = get_book_info(abbr_en)
        print(f"\n📖 开始导入书卷：{name_en}（{abbr_en}），共 {max_chapter} 章")

        chapters = find_chapter_files(abbr_en, max_chapter)
        total_inserted = 0
        skipped = 0

        for idx, (chapter, filepath) in enumerate(chapters, start=1):
            if is_chapter_imported(abbr_en, chapter):
                print(f"[{idx}/{len(chapters)}] ⏭️ 跳过 {abbr_en} {chapter}")
                skipped += 1
                continue

            print(f"[{idx}/{len(chapters)}] 📥 导入 {abbr_en} {chapter} ...", end=" ")
            count = import_chapter(book_id, abbr_en, chapter, filepath)
            total_inserted += count
            print(f"✅ +{count} 节")

        print("\n🎉 导入完成！")
        print(f"   ➕ 新增经文：{total_inserted} 节")
        print(f"   ⏭️ 跳过已导入：{skipped} 章")

    except Exception as e:
        print(f"\n❌ 错误：{e}")

In [14]:
# 从 outputs/plaintext 遍历所有纯文本导入英文，增量更新

import sqlite3
import os
import re

# ========== 配置 ==========
DB_PATH = "db/bible.db"
PLAINTEXT_DIR = "outputs/plaintext"

# ========== 数据库初始化 ==========
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS import_progress (
    abbr_en TEXT,
    chapter INTEGER,
    done INTEGER DEFAULT 0,
    PRIMARY KEY (abbr_en, chapter)
)
""")
conn.commit()

# ========== 工具函数 ==========
def get_book_id(abbr_en):
    cursor.execute(
        "SELECT id FROM book WHERE abbr_en = ?",
        (abbr_en,)
    )
    row = cursor.fetchone()
    if not row:
        raise ValueError(f"❌ 数据库中未找到书卷缩写：{abbr_en}")
    return row[0]


def is_chapter_imported(abbr_en, chapter):
    cursor.execute(
        "SELECT 1 FROM import_progress WHERE abbr_en = ? AND chapter = ? AND done = 1",
        (abbr_en, chapter)
    )
    return cursor.fetchone() is not None


def mark_chapter_done(abbr_en, chapter):
    cursor.execute("""
        INSERT INTO import_progress (abbr_en, chapter, done)
        VALUES (?, ?, 1)
        ON CONFLICT(abbr_en, chapter) DO UPDATE SET done = 1
    """, (abbr_en, chapter))
    conn.commit()


def import_chapter(abbr_en, chapter, filepath):
    book_id = get_book_id(abbr_en)

    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    inserted = 0

    for i, text_en in enumerate(lines, start=1):
        verse_id = f"{abbr_en}.{chapter}.{i}"

        cursor.execute(
            "SELECT 1 FROM verse WHERE id = ?",
            (verse_id,)
        )
        if cursor.fetchone():
            continue

        cursor.execute("""
            INSERT INTO verse (id, book_id, chapter, verse, text_en, text_cn)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (verse_id, book_id, chapter, i, text_en, ""))
        inserted += 1

    conn.commit()
    mark_chapter_done(abbr_en, chapter)
    return inserted


# ========== 主流程 ==========
if __name__ == "__main__":
    # 匹配：{book_abbr}_{chapter}_en.txt
    pattern = re.compile(r"^([A-Za-z0-9]+)_(\d+)_en\.txt$")

    files = []
    for filename in os.listdir(PLAINTEXT_DIR):
        match = pattern.match(filename)
        if not match:
            continue
        abbr_en = match.group(1)
        chapter = int(match.group(2))
        files.append((abbr_en, chapter, os.path.join(PLAINTEXT_DIR, filename)))

    if not files:
        print("❌ 未找到任何匹配的 txt 文件")
    else:
        total_inserted = 0
        skipped = 0

        for idx, (abbr_en, chapter, filepath) in enumerate(files, start=1):
            if is_chapter_imported(abbr_en, chapter):
                print(f"[{idx}/{len(files)}] ⏭️ 跳过 {abbr_en} {chapter}")
                skipped += 1
                continue

            print(f"[{idx}/{len(files)}] 📥 导入 {abbr_en} {chapter} ...", end=" ")
            count = import_chapter(abbr_en, chapter, filepath)
            total_inserted += count
            print(f"✅ +{count} 节")

        print("\n🎉 全部导入完成！")
        print(f"   ➕ 新增经文：{total_inserted} 节")
        print(f"   ⏭️ 跳过已导入：{skipped} 章")

[1/1100] 📥 导入 Jer 26 ... ✅ +24 节
[2/1100] 📥 导入 Mk 5 ... ✅ +43 节
[3/1100] 📥 导入 Rom 8 ... ✅ +39 节
[4/1100] 📥 导入 Rev 20 ... ✅ +15 节
[5/1100] 📥 导入 Ezk 46 ... ✅ +24 节
[6/1100] 📥 导入 Am 7 ... ✅ +17 节
[7/1100] 📥 导入 2S 4 ... ✅ +12 节
[8/1100] 📥 导入 Jer 38 ... ✅ +28 节
[9/1100] 📥 导入 Lev 9 ... ✅ +24 节
[10/1100] 📥 导入 2K 2 ... ✅ +25 节
[11/1100] 📥 导入 2Chr 9 ... ✅ +31 节
[12/1100] 📥 导入 Pro 4 ... ✅ +27 节
[13/1100] 📥 导入 Zec 9 ... ✅ +17 节
[14/1100] 📥 导入 Ezk 25 ... ✅ +17 节
[15/1100] 📥 导入 Wis 12 ... ✅ +27 节
[16/1100] 📥 导入 Is 14 ... ✅ +32 节
[17/1100] 📥 导入 Jer 45 ... ✅ +5 节
[18/1100] 📥 导入 Num 10 ... ✅ +36 节
[19/1100] 📥 导入 2Chr 5 ... ✅ +14 节
[20/1100] 📥 导入 Is 65 ... ✅ +25 节
[21/1100] 📥 导入 Jer 34 ... ✅ +22 节
[22/1100] 📥 导入 2S 8 ... ✅ +18 节
[23/1100] 📥 导入 2S 16 ... ✅ +23 节
[24/1100] 📥 导入 Is 18 ... ✅ +7 节
[25/1100] 📥 导入 1K 13 ... ✅ +34 节
[26/1100] 📥 导入 1Cor 15 ... ✅ +58 节
[27/1100] 📥 导入 Jer 49 ... ✅ +39 节
[28/1100] 📥 导入 Zec 5 ... ✅ +11 节
[29/1100] 📥 导入 Ezk 29 ... ✅ +21 节
[30/1100] 📥 导入 Pro 8 ... ✅ +36 节
[31/1100] 📥

[281/1100] 📥 导入 Ps 4 ... ✅ +9 节
[282/1100] 📥 导入 Rev 18 ... ✅ +24 节
[283/1100] 📥 导入 Mt 13 ... ✅ +58 节
[284/1100] 📥 导入 Num 36 ... ✅ +13 节
[285/1100] 📥 导入 Gen 6 ... ✅ +22 节
[286/1100] 📥 导入 Is 32 ... ✅ +20 节
[287/1100] 📥 导入 Is 51 ... ✅ +23 节
[288/1100] 📥 导入 2S 22 ... ✅ +51 节
[289/1100] 📥 导入 Rev 7 ... ✅ +17 节
[290/1100] 📥 导入 Jdg 3 ... ✅ +31 节
[291/1100] 📥 导入 Num 28 ... ✅ +31 节
[292/1100] 📥 导入 2Cor 10 ... ✅ +18 节
[293/1100] 📥 导入 Mt 9 ... ✅ +38 节
[294/1100] 📥 导入 Hos 9 ... ✅ +17 节
[295/1100] 📥 导入 Lk 10 ... ✅ +42 节
[296/1100] 📥 导入 Ex 29 ... ✅ +46 节
[297/1100] 📥 导入 Ps 136 ... ✅ +26 节
[298/1100] 📥 导入 Gen 18 ... ✅ +33 节
[299/1100] 📥 导入 Phil 2 ... ✅ +30 节
[300/1100] 📥 导入 Dt 20 ... ✅ +20 节
[301/1100] 📥 导入 1Chr 21 ... ✅ +30 节
[302/1100] 📥 导入 Gal 2 ... ✅ +21 节
[303/1100] 📥 导入 2Chr 10 ... ✅ +19 节
[304/1100] 📥 导入 Job 33 ... ✅ +33 节
[305/1100] 📥 导入 Ob 1 ... ✅ +21 节
[306/1100] 📥 导入 1Mac 10 ... ✅ +89 节
[307/1100] 📥 导入 Lk 9 ... ✅ +62 节
[308/1100] 📥 导入 1Jn 1 ... ✅ +10 节
[309/1100] 📥 导入 Ex 37 ... ✅ +29 节
[310

[579/1100] 📥 导入 Job 26 ... ✅ +14 节
[580/1100] 📥 导入 Gen 13 ... ✅ +18 节
[581/1100] 📥 导入 2Mac 7 ... ✅ +42 节
[582/1100] 📥 导入 Ps 140 ... ✅ +14 节
[583/1100] 📥 导入 Rom 14 ... ✅ +23 节
[584/1100] 📥 导入 Sir 6 ... ✅ +37 节
[585/1100] 📥 导入 Ps 115 ... ✅ +18 节
[586/1100] 📥 导入 Jn 15 ... ✅ +27 节
[587/1100] 📥 导入 2K 21 ... ✅ +26 节
[588/1100] 📥 导入 Lev 14 ... ✅ +57 节
[589/1100] 📥 导入 1P 1 ... ✅ +25 节
[590/1100] 📥 导入 Pro 21 ... ✅ +31 节
[591/1100] 📥 导入 Zep 2 ... ✅ +15 节
[592/1100] 📥 导入 Mk 14 ... ✅ +72 节
[593/1100] 📥 导入 Gen 46 ... ✅ +34 节
[594/1100] 📥 导入 Job 10 ... ✅ +22 节
[595/1100] 📥 导入 1Cor 8 ... ✅ +13 节
[596/1100] 📥 导入 2Chr 33 ... ✅ +25 节
[597/1100] 📥 导入 Job 9 ... ✅ +35 节
[598/1100] 📥 导入 Gen 25 ... ✅ +34 节
[599/1100] 📥 导入 Ps 39 ... ✅ +14 节
[600/1100] 📥 导入 Ex 14 ... ✅ +31 节
[601/1100] 📥 导入 Heb 13 ... ✅ +25 节
[602/1100] 📥 导入 1Thes 1 ... ✅ +10 节
[603/1100] 📥 导入 Dt 5 ... ✅ +33 节
[604/1100] 📥 导入 Ps 44 ... ✅ +27 节
[605/1100] 📥 导入 Num 2 ... ✅ +34 节
[606/1100] 📥 导入 Ecl 12 ... ✅ +14 节
[607/1100] 📥 导入 Jn 7 ... ✅ +53 节

[888/1100] 📥 导入 Jer 17 ... ✅ +27 节
[889/1100] 📥 导入 Jer 2 ... ✅ +37 节
[890/1100] 📥 导入 Is 58 ... ✅ +14 节
[891/1100] 📥 导入 1K 2 ... ✅ +46 节
[892/1100] 📥 导入 Jdg 14 ... ✅ +20 节
[893/1100] 📥 导入 Ezk 14 ... ✅ +23 节
[894/1100] 📥 导入 Ps 95 ... ✅ +11 节
[895/1100] 📥 导入 Num 21 ... ✅ +35 节
[896/1100] 📥 导入 1S 4 ... ✅ +22 节
[897/1100] 📥 导入 Is 25 ... ✅ +12 节
[898/1100] 📥 导入 Ps 42 ... ✅ +12 节
[899/1100] 📥 导入 Num 4 ... ✅ +49 节
[900/1100] 📥 导入 Jn 1 ... ✅ +51 节
[901/1100] 📥 导入 Dt 3 ... ✅ +29 节
[902/1100] 📥 导入 Ex 12 ... ✅ +51 节
[903/1100] 📥 导入 Job 16 ... ✅ +22 节
[904/1100] 📥 导入 2Chr 35 ... ✅ +27 节
[905/1100] 📥 导入 Gen 23 ... ✅ +20 节
[906/1100] 📥 导入 1S 22 ... ✅ +23 节
[907/1100] 📥 导入 Gen 40 ... ✅ +23 节
[908/1100] 📥 导入 Pro 27 ... ✅ +27 节
[909/1100] 📥 导入 Mk 12 ... ✅ +44 节
[910/1100] 📥 导入 Lev 12 ... ✅ +8 节
[911/1100] 📥 导入 Ezk 6 ... ✅ +14 节
[912/1100] 📥 导入 Ps 21 ... ✅ +14 节
[913/1100] 📥 导入 Ps 113 ... ✅ +9 节
[914/1100] 📥 导入 Jos 19 ... ✅ +51 节
[915/1100] 📥 导入 Jn 13 ... ✅ +38 节
[916/1100] 📥 导入 1Tim 3 ... ✅ +16 节
[917/1